In [1]:
import os
from typing import List
from neo4j import GraphDatabase

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from langchain_community.graphs import Neo4jGraph
from neo4j import GraphDatabase

os.environ["NEO4J_URI"] = ""
os.environ["NEO4J_USERNAME"] = ""
os.environ["NEO4J_PASSWORD"] = ""

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(
        os.environ["NEO4J_USERNAME"],
        os.environ["NEO4J_PASSWORD"]
    )
)

In [3]:
with driver.session() as session:
    result = session.run("MATCH (n) RETURN count(n)")
    print(result.single()[0])


684


In [4]:
from eval_utils import load_embeddings

embedding = load_embeddings(model_id="sentence-transformers/all-MiniLM-L6-v2",
                            cache_dir="hf_models/embeddings/all-MiniLM-L6-v2")


/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/RAGAS/eval_utils.py:55: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


Load Sentence Nodes from Neo4j

In [5]:
def load_sentences_from_neo4j():
    cypher = """
    MATCH (s:Sentence)
    RETURN s.id AS id,
           s.text AS text,
           s.type AS type,
           s.source AS source
    """
    docs = []

    with driver.session() as session:
        for r in session.run(cypher):
            docs.append(
                Document(
                    page_content=r["text"],
                    metadata={
                        "id": r["id"],
                        "type": r["type"],
                        "source": r["source"]
                    }
                )
            )
    return docs


sentence_docs = load_sentences_from_neo4j()

len(sentence_docs), sentence_docs[0]


(617,
 Document(metadata={'id': 0, 'type': 'solution', 'source': '../data/mk4s_manuel/Extruder_blob.md'}, page_content='Extruder blob\n=============\n\nThe blob (mass of plastic accumulated around the hotend) is one of the most scary-looking printing problems you might face with your 3D printer.'))

In [6]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    sentence_docs,
    embedding
)


In [7]:
vectorstore.similarity_search(
    "How to fix preheat error?",
    k=3
)


[Document(id='a9350843-933b-411a-8b5c-51bb51c6cd42', metadata={'id': 41, 'type': None, 'source': '../data/mk4s_manuel/Preheat_error.md'}, page_content='Troubleshooting\n\nPreheat error\n\n1.'),
 Document(id='1286dd77-652b-431d-a8dd-8f29cd3633a8', metadata={'id': 39, 'type': None, 'source': '../data/mk4s_manuel/Preheat_error.md'}, page_content='Preheat error\xa0- a problem with preheating the hotend.'),
 Document(id='ea37523b-0704-4087-93f3-5682a631497e', metadata={'id': 50, 'type': None, 'source': '../data/mk4s_manuel/Preheat_error.md'}, page_content='Bed preheat error\n\n1.')]

Entities retrieval

In [21]:
def graph_retrieve(question: str, top_k: int = 6) -> List[Document]:
    q = question.lower()

    tokens = [t for t in q.replace("?", " ").replace(",", " ").split() if len(t) >= 3]
    cypher = """
    WITH $tokens AS tokens
    MATCH (e:Entity)
    WHERE any(t IN tokens WHERE e.name CONTAINS t)
    MATCH (s:Sentence)-[:MENTIONS]->(e)
    OPTIONAL MATCH (d:Document)-[:CONTAINS]->(s)
    WITH s, d,
         CASE s.type WHEN 'solution' THEN 3 WHEN 'fix' THEN 3 WHEN 'cause' THEN 1 ELSE 0 END AS score
    RETURN s.text AS text,
           s.type AS type,
           coalesce(d.source, '') AS source,
           score
    ORDER BY score DESC
    LIMIT $top_k
    """

    docs = []
    with driver.session() as session:
        rows = session.run(cypher, tokens=tokens, top_k=top_k).data()

    for r in rows:
        docs.append(
            Document(
                page_content=r["text"],
                metadata={
                    "type": r["type"],
                    "source": r["source"],
                    "retrieval": "graph"
                }
            )
        )
    return docs


In [22]:
def embedding_retrieve(question: str, top_k: int = 4) -> List[Document]:
    return vectorstore.similarity_search(question, k=top_k)


Hybrid Retrieval：Vector Search + Graph Search

In [36]:
def hybrid_retrieve(question: str,
                    graph_k: int = 15,
                    embed_k: int = 15) -> List[Document]:

    graph_docs = graph_retrieve(question, top_k=graph_k)


    if len(graph_docs) >= graph_k:
        return graph_docs

    embed_docs = embedding_retrieve(question, top_k=embed_k)

    seen = set(d.page_content for d in graph_docs)
    merged = graph_docs[:]

    for d in embed_docs:
        if d.page_content not in seen:
            merged.append(d)
            seen.add(d.page_content)

    return merged


In [37]:
retriever = RunnableLambda(lambda q: hybrid_retrieve(q))

Context Formatting & Prioritization

In [38]:
def format_docs(docs: List[Document]) -> str:
    def rank(d):
        t = (d.metadata.get("type") or "").lower()
        return 0 if t in ("solution", "fix") else 1

    docs = sorted(docs, key=rank)

    lines = []
    for i, d in enumerate(docs, 1):
        lines.append(
            f"[{i}] ({d.metadata.get('type')}, via {d.metadata.get('retrieval')})\n"
            f"{d.page_content}\n"
            f"source: {d.metadata.get('source')}"
        )
    return "\n\n".join(lines)


In [39]:
from dotenv import load_dotenv
import getpass
import os
from eval_utils import load_llm

if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your token:")

load_dotenv()


chat_model = load_llm(model_path="./hf_models/Llama-3.1-8B-Instruct")

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.30it/s]
Device set to use cuda:0


In [40]:
from langchain_core.prompts import PromptTemplate

template = """You are a helpful assistant specializing in 3D printing operations.
Use the following pieces of retrieved context to answer the question.
If you don’t know the answer, just say that you don’t know.
Use two sentences maximum and keep the answer concise.

Question: {question}
Context: {context}
Answer:
"""
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

prompt = PromptTemplate.from_template(template)

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | chat_model
    | StrOutputParser()
)


In [41]:
(retriever | RunnableLambda(format_docs)).invoke(
    "What are the recommended steps for checking belt tension and ensuring proper motor pulley alignment on both the Original Prusa MK-series and Original Prusa XL printers?"
)

"Make sure that the first layer is sticking properly to the entire print surface.\n\nFor example, if the blob is made of:\n   \n   * PLA:\xa0preheat to 250°C\n   * ABS/PETG:\xa0preheat to 280°C\n\n2.\n\n6.\n\n_Turn the printer off before the next step, or you might cause an electrical short.\n\nAlways make sure to\xa0clean the print surface\xa0before starting the print.\n\nFor example, if the blob is made of:\n   \n   * PLA:\xa0preheat to 250°C\n   * ABS/PETG:\xa0preheat to 280°C\n\n2.\n\n6.\n\nWork very carefully, you don't want to melt the printed parts surrounding the hotend.\n\nHow to prevent this problem from happening\n\nUnlike the\xa0Spaghetti monster\xa0issue, the blob issue often occurs early during the print.\n\nIf\xa0MINTEMP\xa0or\xa0Preheat error\xa0message appears on the LCD, it means the thermistor or the heater cable (respectively) is already broken.\n\nIn case you do not succeed, as well as in case the blob has damaged some of the extruder parts, you can find your spare

In [42]:
rag_chain.invoke("What are the recommended steps for checking belt tension and ensuring proper motor pulley alignment on both the Original Prusa MK-series and Original Prusa XL printers?")

'To check belt tension and ensure proper motor pulley alignment on both the Original Prusa MK-series and Original Prusa XL printers, turn the printer off, then loosen the belt tension screws and align the pulleys while checking for any obstructions or blockages.'

Build Chain and Evaluation

In [43]:
import pandas as pd
import time
from ragas import EvaluationDataset
from ragas import evaluate, RunConfig
from ragas.metrics import LLMContextPrecisionWithReference, LLMContextRecall, AnswerAccuracy, AnswerRelevancy, Faithfulness, FactualCorrectness
from eval_utils import load_eval_model


def get_eval_ds(path, retriever, rag_chain):

    testset = pd.read_csv(path)
    queries = testset["user_input"].to_list()
    references = testset["reference"].to_list()

    # generate evaluation dataset
    rs_times = []
    dataset = []

    for query, reference in zip(queries, references):

        
        relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
        # measure response time
        t0 = time.perf_counter()
        response = rag_chain.invoke(query)
        t1 = time.perf_counter()
        
        dataset.append(
            {
                "user_input": query,
                "retrieved_contexts": relevant_docs,
                "response": response,
                "reference": reference,
            }
        )
        rs_times.append(t1 -t0)

    eval_ds = EvaluationDataset.from_list(dataset)

    return eval_ds, rs_times

def get_eval_result(eval_ds, metrics):
    evaluator_llm = load_eval_model()

    run_config = RunConfig(
    timeout=600,      
    max_workers=2,    
    max_retries=2,    
    max_wait=600,     
)
    result = evaluate(
        dataset=eval_ds,
        metrics=metrics,
        llm=evaluator_llm,
        run_config=run_config,
        )
    
    return result

metrics=[LLMContextPrecisionWithReference(),
         LLMContextRecall(), 
         AnswerAccuracy(),
         AnswerRelevancy(),
         Faithfulness(),
         FactualCorrectness(mode = 'f1', atomicity='high', coverage='high')]

In [44]:
data_path = "evaluate_dataset/test_dataset.csv"

eval_ds, rs_times = get_eval_ds(data_path, retriever, rag_chain)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [45]:
result = get_eval_result(eval_ds, metrics)
result

Evaluating: 100%|██████████| 240/240 [30:55<00:00,  7.73s/it]


{'llm_context_precision_with_reference': 0.1835, 'context_recall': 0.5739, 'nv_accuracy': 0.4125, 'answer_relevancy': 0.9160, 'faithfulness': 0.6375, 'factual_correctness(mode=f1)': 0.3443}

In [ ]:
df = result.to_pandas()
df['response_time'] = rs_times
df.to_csv("./evaluate_results/09_KG_test/rule_based_test.csv")